In [28]:
#13.Start from the raw Titanic dataset rather than your cleaned copy, so the pipeline has real work to do.
#14.List your numeric columns and your categorical columns separately.

import seaborn as sb
import pandas as pd
titanic=sb.load_dataset("titanic")
X = titanic.drop(["survived","alive"], axis=1)
y = titanic["survived"]
numerical_columns = X.select_dtypes(include=["int64", "float64"]).columns
categorical_columns = X.select_dtypes(include=["object","string", "category"]).columns
print(f"Numerical Columns: {numerical_columns}")
print(f"Categorical Columns: {categorical_columns}")


Numerical Columns: Index(['pclass', 'age', 'sibsp', 'parch', 'fare'], dtype='str')
Categorical Columns: Index(['sex', 'embarked', 'class', 'who', 'deck', 'embark_town'], dtype='str')


In [29]:
#15.Build one small pipeline for the numeric columns that imputes missing values 
#and then standardizes them, and another for the categorical columns that imputes
#and then one hot encodes them.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
numerical_pipeline=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])
categorical_pipeline=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown="ignore"))])


In [30]:
#16.Combine both with a ColumnTransformer, then place that transformer
#and your classifier together inside a single Pipeline.
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
preprocessor=ColumnTransformer([("numeric",numerical_pipeline,numerical_columns),("categorical",categorical_pipeline,categorical_columns)])
model_pipeline=Pipeline([("preprocessor",preprocessor),("classifier",LogisticRegression())])

In [31]:
#17.Run cross_val_score on the whole pipeline and compare the result against 
#the score you reached in Week Two using manual steps.
from sklearn.model_selection import cross_val_score
x = titanic.drop("survived", axis=1)
y = titanic["survived"]
scores = cross_val_score(model_pipeline,x,y,cv=5)
print("CV Scores:", scores)
print("Mean CV Score:", scores.mean())

CV Scores: [0.82122905 0.79213483 0.80898876 0.80337079 0.84269663]
Mean CV Score: 0.8136840123030569


In [32]:
#18.Fit the pipeline once, then hand it two or three invented passenger rows and 
#confirm it predicts with no preparation on your part at all.
model_pipeline.fit(X, y)
new_passengers = pd.DataFrame({
    "pclass": [1, 3, 2],
    "sex": ["female", "male", "female"],
    "age": [25, 30, 40],
    "sibsp": [0, 1, 0],
    "parch": [0, 0, 1],
    "fare": [80.0, 8.0, 20.0],
    "embarked": ["S", "S", "C"],
    "class": ["First", "Third", "Second"],
    "who": ["woman", "man", "woman"],
    "adult_male": [False, True, False],
    "deck": ["C", None, "B"],
    "embark_town": ["Southampton", "Southampton", "Cherbourg"],
    "alone": [True, False, False]
})
new_passengers = new_passengers[X.columns]
predictions = model_pipeline.predict(new_passengers)
print("New Passengers:")
print(new_passengers)
print("\nPredictions:")
print(predictions)

New Passengers:
   pclass     sex  age  sibsp  parch  fare embarked   class    who  \
0       1  female   25      0      0  80.0        S   First  woman   
1       3    male   30      1      0   8.0        S   Third    man   
2       2  female   40      0      1  20.0        C  Second  woman   

   adult_male deck  embark_town  alone  
0       False    C  Southampton   True  
1        True  NaN  Southampton  False  
2       False    B    Cherbourg  False  

Predictions:
[1 0 1]
